In [ ]:
pip install gurobipy

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np

In [ ]:
# Create a new Gurobi model
model = gp.Model("MarksCafe")

# Data

type_Tables = 2
type_Labor = 2
laborCost = np.array([18,32])
laborMinutes = np.array([[10,15],[20,35]])
RevenuePerHour = np.array([100,150])
MaxNoTables = np.array([6,12])
maxLaborHours = np.array([480,300])

# Create the decision variable
x ={}


In [ ]:
for j in range(type_Tables):
    x[j] = model.addVar(vtype=GRB.INTEGER, name=f"x_{j}")

In [ ]:
# Set objective: Maximize Profit
model.setObjective(gp.quicksum((RevenuePerHour[j] - laborMinutes[0,j]/60 * laborCost[0] - laborMinutes[1,j]/60 * laborCost[1]) * x[j] for j in range(type_Tables)), GRB.MAXIMIZE,)

In [ ]:
# Constraints

# Ensure Max number of Tables
for c in range(type_Tables):
    model.addConstr(x[c] <= MaxNoTables[c], f"MaxTables{c}")

# Ensure Max Labor minutes
for j in range(type_Labor):
    model.addConstr(gp.quicksum(laborMinutes[j,c] * x[j] for c in range(type_Tables)) <= maxLaborHours[j], f'Max labor hours')

In [ ]:
# Optimize the model
model.optimize()

# Print the results
if model.status == GRB.OPTIMAL:
    print("Optimal Soltuion Found:")
    for j in range(type_Tables):
        if x[j].x > 0:
            print(f"Tables {j + 1}", x[j].x)
            
        print(f"Total Profit: ${model.objVal:.2f}")
else:
    print("No solution found.")